<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Vantage In-database Graph Analysis: PageRank
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>What is PageRank:</b></p>
<p style = 'font-size:24px;font-family:Arial;color:#00233C'></p>PageRank function measures the importance or influence of each vertex (node) in a graph based on the structure of incoming edges</p>

Key features:
<ul style = 'font-size:16px;font-family:Arial;color:#00233C'>
    <li>Support directional and un-directional network</li>
    <li>Work with weigthed and unweighted network</li>
    <li>Supported by Teradata MPP architecture</li>
    <li>Real-time analytics</li>
</ul>
<hr>

In [ ]:
# Teradata python package
from teradataml import *
from TeradataGE import td_graph_function, configure

import pandas as pd
import json
import getpass
import numpy as np

# for local display only
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import time


In [ ]:
configure.graph_install_location = "GraphLib"

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Connect to Teradata Vantage Instance</b></p>

In [ ]:
hostname = "xx.xx.xx.xx"
user = input(prompt=f"Username for database:")
password = getpass.getpass(prompt=f"Database password for {user}:")
databasename = "graphdb"
logmech = "TD2"

eng = create_context(host = hostname, username = user, password = password, database= databasename, logmech = logmech)
print(eng)

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Run All Four Variants</b></p>
<p style = 'font-size:20px;font-family:Arial;color:#00233C'><b>Variant 1: Directed + Unweighted</b></p>


In [ ]:
graph_obj = td_graph_function.td_graph_object(edge_table_name = "graph_edges",
                                              edge_from_node_column_name = "source",
                                              edge_to_node_column_name = "target",
                                              edge_weight_column_name = None)
t_start = time.time()
print("Running: Directed + Unweighted")
df_du, iters_du = graph_obj.td_pagerank()
elapsed = time.time() - t_start
df_du = df_du.to_pandas().reset_index().rename(columns={"pr_score": "pr_directed_unweighted"})
print(f"Wall time: {elapsed:.2f}s")
print(f"Shape: {df_du.shape}")
df_du.head(10)

<p style = 'font-size:20px;font-family:Arial;color:#00233C'><b>Variant 2: Directed + Weighted</b></p>

In [ ]:
graph_obj = td_graph_function.td_graph_object(edge_table_name = "graph_edges",
                                              edge_from_node_column_name = "source",
                                              edge_to_node_column_name = "target",
                                              edge_weight_column_name = "weight")
t_start = time.time()
print("Running: Directed + Weighted")
df_dw, iters_dw = graph_obj.td_pagerank()
elapsed = time.time() - t_start
df_dw = df_dw.to_pandas().reset_index().rename(columns={"pr_score": "pr_directed_weighted"})
print(f"Wall time: {elapsed:.2f}s")
df_dw.head(10)

<p style = 'font-size:20px;font-family:Arial;color:#00233C'><b>Variant 3: Undirected + Unweighted</b></p>

In [ ]:
graph_obj = td_graph_function.td_graph_object(edge_table_name = "graph_edges",
                                              edge_from_node_column_name = "source",
                                              edge_to_node_column_name = "target",
                                              edge_weight_column_name = None)
t_start = time.time()
print("Running: Undirected + Unweighted")
df_uu, iters_uu = graph_obj.td_pagerank(directed=False)
elapsed = time.time() - t_start
df_uu = df_uu.to_pandas().reset_index().rename(columns={"pr_score": "pr_undirected_unweighted"})
print(f"Wall time: {elapsed:.2f}s")
df_uu.head(10)

<p style = 'font-size:20px;font-family:Arial;color:#00233C'><b>Variant 4: Undirected + Weighted</b></p>

In [ ]:
graph_obj = td_graph_function.td_graph_object(edge_table_name = "graph_edges",
                                              edge_from_node_column_name = "source",
                                              edge_to_node_column_name = "target",
                                              edge_weight_column_name = "weight")
t_start = time.time()
print("Running: Undirected + Weighted")
df_uw, iters_uw = graph_obj.td_pagerank(directed=False)
elapsed = time.time() - t_start
df_uw = df_uw.to_pandas().reset_index().rename(columns={"pr_score": "pr_undirected_weighted"})
print(f"Wall time: {elapsed:.2f}s")
df_uw.head(10)

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Convergence Summary</b></p>

In [ ]:
print("Iterations to converge (tolerance = 1e-8):")
print(f"  Directed + Unweighted:   {iters_du}")
print(f"  Directed + Weighted:     {iters_dw}")
print(f"  Undirected + Unweighted: {iters_uu}")
print(f"  Undirected + Weighted:   {iters_uw}")

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Compare All Four Variants</b></p>

In [ ]:
df_all = df_du[["node", "pr_directed_unweighted"]].copy()
df_all = df_all.merge(df_dw[["node", "pr_directed_weighted"]], on="node")
df_all = df_all.merge(df_uu[["node", "pr_undirected_unweighted"]], on="node")
df_all = df_all.merge(df_uw[["node", "pr_undirected_weighted"]], on="node")

df_all = df_all.sort_values("pr_directed_unweighted", ascending=False).reset_index(drop=True)
df_all.head(15)

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Visualisations</b></p>

In [ ]:
top_n = 20
top_nodes = df_all.head(top_n)["node"].values

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"PageRank Scores (Stored Procedure, Optimised) - Top {top_n} Nodes", fontsize=16)

variants = [
    ("pr_directed_unweighted", "Directed + Unweighted"),
    ("pr_directed_weighted", "Directed + Weighted"),
    ("pr_undirected_unweighted", "Undirected + Unweighted"),
    ("pr_undirected_weighted", "Undirected + Weighted"),
]

for ax, (col, title) in zip(axes.flatten(), variants):
    data = df_all[df_all["node"].isin(top_nodes)].sort_values(col, ascending=True)
    ax.barh(data["node"].astype(str), data[col], color="steelblue")
    ax.set_title(title)
    ax.set_xlabel("PageRank Score")
    ax.set_ylabel("Node")

plt.tight_layout()
plt.savefig("pagerank_sp_v2_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(df_all["pr_directed_unweighted"], df_all["pr_undirected_unweighted"],
                alpha=0.4, s=10, color="steelblue")
axes[0].plot([0, df_all["pr_directed_unweighted"].max()],
             [0, df_all["pr_directed_unweighted"].max()], "r--", alpha=0.5)
axes[0].set_xlabel("Directed Unweighted")
axes[0].set_ylabel("Undirected Unweighted")
axes[0].set_title("Directed vs Undirected (Unweighted)")

axes[1].scatter(df_all["pr_directed_weighted"], df_all["pr_undirected_weighted"],
                alpha=0.4, s=10, color="darkorange")
axes[1].plot([0, df_all["pr_directed_weighted"].max()],
             [0, df_all["pr_directed_weighted"].max()], "r--", alpha=0.5)
axes[1].set_xlabel("Directed Weighted")
axes[1].set_ylabel("Undirected Weighted")
axes[1].set_title("Directed vs Undirected (Weighted)")

plt.tight_layout()
plt.savefig("pagerank_sp_v2_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Rank Correlation Between Variants</b></p>

In [ ]:
from scipy.stats import spearmanr

pr_cols = ["pr_directed_unweighted", "pr_directed_weighted",
           "pr_undirected_unweighted", "pr_undirected_weighted"]

corr_matrix = pd.DataFrame(index=pr_cols, columns=pr_cols, dtype=float)
for c1 in pr_cols:
    for c2 in pr_cols:
        corr, _ = spearmanr(df_all[c1], df_all[c2])
        corr_matrix.loc[c1, c2] = round(corr, 4)

print("Spearman Rank Correlation between PageRank variants:\n")
corr_matrix

<hr>
<p style = 'font-size:28px;font-family:Arial;color:#00233C'><b>Disconnect from Teradata</b></p>

In [ ]:
remove_context()